In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("final_preprocessed.csv")
df.head(5)

,Unnamed: 0,category,error_code,description,remedy,error_type,description_cleaned,remedy_cleaned,content
0,0,Configuration error of the interface - Modbus_...,16#8181,The module does not support this data transmis...,Select a valid data transmission rate for the ...,Modbus,module doe support data transmission rate,Select valid data transmission rate module BAU...,error_code: 16#8181\ndescription: The module d...
1,1,Configuration error of the interface - Modbus_...,16#8182,The module does not support this parity setting.,"Select a suitable value for ""Parity"" at the PA...",Modbus,module doe support parity setting,Select suitable value Parity PARITY parameter ...,error_code: 16#8182\ndescription: The module d...
2,2,Configuration error of the interface - Modbus_...,16#8183,The module does not support this type of data ...,Select a valid data flow control for the modul...,Modbus,module doe support type data flow control,Select valid data flow control module FLOWCTRL...,error_code: 16#8183\ndescription: The module d...
3,3,Configuration error of the interface - Modbus_...,16#8184,"Invalid value for ""Response timeout"".","Select a suitable value for ""Response timeout""...",Modbus,Invalid value Response timeout,Select suitable value Response timeout RESPTO ...,error_code: 16#8184\ndescription: Invalid valu...
4,4,Configuration error of the interface - Modbus_...,16#8280,Negative acknowledgment when reading module,Check the input at the PORT parameter.\nYou ca...,Modbus,Negative acknowledgment reading module,Check input PORT parameter find detailed infor...,error_code: 16#8280\ndescription: Negative ack...


In [3]:
df['content'][0]

'error_code: 16#8181\ndescription: The module does not support this data transmission rate.\nremedy: Select a valid data transmission rate for the module at the BAUD parameter.'

In [6]:
df.shape

(167, 9)

In [22]:
from sentence_transformers import SentenceTransformer

def generate_embeddings(data):
    # Initialize the SentenceTransformer model
    model = SentenceTransformer('all-MiniLM-L6-v2')  # You can choose other pre-trained models
    
    # Generate embeddings for the data
    embeddings = model.encode(data, convert_to_tensor=False).tolist
    
    return embeddings

# Example usage
df['embeddings'] = df['content'].apply(generate_embeddings)



ValueError: setting an array element with a sequence.

In [16]:
df.to_csv('dataWithEmbeddings.csv')

In [24]:
df = pd.read_csv("dataWithEmbeddings.csv")

In [27]:
df['embeddings'][0]

'[0.021754419431090355, -0.002193288877606392, -0.0808594673871994, -0.03987583518028259, -0.048045288771390915, -0.06272785365581512, -0.04114222899079323, 0.003700645873323083, -0.0767945647239685, 0.010286041535437107, 0.10623329132795334, -0.06244420260190964, 0.03647741675376892, 0.01946033537387848, 0.03021319769322872, 0.07917071878910065, 0.06934184581041336, -0.006084586959332228, -0.009992717765271664, 0.06322275847196579, 0.018773768097162247, 0.07377045601606369, -0.00022228418674785644, 0.014114138670265675, 0.034069597721099854, -0.08888255059719086, 0.035503510385751724, 0.006023977883160114, 0.004090093541890383, 0.030872561037540436, 0.025771839544177055, 0.08333957940340042, -0.046728916466236115, 0.041517358273267746, 0.041986461728811264, 0.025030914694070816, -0.014887121506035328, -0.07780038565397263, 0.01026664488017559, 0.010431199334561825, 0.10152186453342438, -0.009592468850314617, -0.03308708593249321, 0.04650246351957321, -0.019654670730233192, -0.03138712

In [31]:
import numpy as np
import ast
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Function to generate embeddings for the query text
def generate_query_embedding(query):
    model = SentenceTransformer('all-MiniLM-L6-v2')  # Pre-trained model
    query_embedding = model.encode([query], convert_to_tensor=False)  # Get embedding for the query
    return query_embedding



In [32]:
# Function to fetch top 5 similar items based on cosine similarity
def fetch_top_5_similar(query, dataset):
    # Generate the query embedding
    query_embedding = generate_query_embedding(query)
    
    # Ensure the query embedding is a numpy array (2D array)
    query_embedding = np.array(query_embedding)
    
    # Preprocess the embeddings: Ensure they are in correct format (as lists of floats)
    dataset_embeddings = []
    for embedding in dataset['embeddings']:
        try:
            # If the embeddings are stored as strings (e.g., '[0.1, 0.2, 0.3]')
            embedding_list = ast.literal_eval(embedding)  # Safely convert string representation to list
            dataset_embeddings.append(np.array(embedding_list, dtype=float))
        except Exception as e:
            print(f"Error processing embedding: {embedding}")
            print(e)

    # Convert to numpy array for cosine similarity calculation
    dataset_embeddings = np.array(dataset_embeddings)

    # Extract contents for matching
    dataset_contents = dataset['content']
    
    # Calculate cosine similarity between the query embedding and dataset embeddings
    similarities = cosine_similarity(query_embedding, dataset_embeddings)[0]
    
    # Get indices of the top 5 most similar items (highest cosine similarity)
    top_5_indices = np.argsort(similarities)[-5:][::-1]  # Sort descending
    
    # Fetch the top 5 most similar data based on the sorted indices
    top_5_data = [(dataset_contents[i], similarities[i]) for i in top_5_indices]
    
    return top_5_data

In [38]:
# Example dataset with content and precomputed embeddings
dataset = df

# Example query
query = "I am getting Negative acknowledgment when reading module what does this mean?"

# Fetch top 5 similar entries based on the query
top_5_data = fetch_top_5_similar(query, dataset)

# Print top 5 most similar data and their cosine similarity scores
for i, (text, similarity) in enumerate(top_5_data):
    print(f"Rank {i+1}: {text} (Cosine Similarity: {similarity:.4f})")

In [36]:
# Example dataset with content and precomputed embeddings
dataset = df

# Example query
query = "what does 16#8280 error code means ?"

# Fetch top 5 similar entries based on the query
top_5_data = fetch_top_5_similar(query, dataset)

# Print top 5 most similar data and their cosine similarity scores
for i, (text, similarity) in enumerate(top_5_data):
    print(f"Rank {i+1}: {text} (Cosine Similarity: {similarity:.4f})")

Rank 1: error_code: 16#80E2
description: Frame aborted: Character frame error
remedy: Check the settings for start bit, data bits, parity bit, data transmission rate, and stop bit(s). (Cosine Similarity: 0.6277)
Rank 2: error_code: 16#818F
description: Incorrect parameter number setting (with USS only)
remedy: Select a suitable parameter number (PARAM).
The following numbers are valid: 0-2047 (Cosine Similarity: 0.6170)
Rank 3: error_code: 16#80E3
description: Frame aborted: Character overflow error
remedy: Check the number of data in the frame of the communication partner. (Cosine Similarity: 0.6092)
Rank 4: error_code: 16#7002
description: Interim call: Data transmission running
remedy: No Remedy (Cosine Similarity: 0.6074)
Rank 5: error_code: 16#8191
description: Incorrect setting of the diagnostic error interrupt
remedy: Select a suitable value for "Diagnostic error interrupt".
The following are valid: Diagnostic error interrupt deactivated or diagnostic error interrupt activated.
